# Séance 4 (suite) : Introduction à MNE-Python

PSY2007D — Laboratoire 1, automne 2026

Les objets de MNE utilisés dans tous les notebooks d'analyse :

```
Raw ──filtrage──▶ Raw filtré ──événements──▶ Epochs ──moyenne──▶ Evoked
 │                                                                  │
spectre (PSD)                                                cartes topographiques
```

| Partie | Contenu |
|---|---|
| 1 | Construire un signal continu `Raw` |
| 2 | `Raw` et `info` |
| 3 | Spectre de puissance et filtrage |
| 4 | Événements et `Epochs` |
| 5 | `Evoked` et cartes topographiques |

> **Données.** Jeu public *kiloword* (Dufau et al., 2015), comme dans `seance4_outils_python.ipynb`. Ce ne sont pas les données du projet. Ce jeu est fourni déjà découpé en essais : la partie 1 les recolle en un signal continu et y **ajoute** du bruit, pour s'exercer.

**Blocs** — Type 1 : exécuter et lire · Type 2 : compléter les lignes marquées `<---` · Type 3 : optionnel.

<details><summary>Colab ou VS Code ?</summary>

- **Colab** : `Fichier → Enregistrer une copie dans Drive` avant de commencer.
- **VS Code** : ouvrir le notebook et choisir le noyau `env_meeg` (en haut à droite).
- Les cellules grises repliées (dans Colab) contiennent du code utilitaire : il suffit de les exécuter.
</details>

Documentation : [mne.tools](https://mne.tools/stable/).

In [ ]:
# ---- Installation (Colab seulement) et imports
import sys
if "google.colab" in sys.modules:
    !pip install -q mne

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

mne.set_log_level("WARNING")
print("MNE :", mne.__version__)

## 1. Construire un signal continu `Raw`

### Bloc Type 1 — Charger kiloword

Environ 25 Mo la première fois. Sans connexion, la cellule crée des données simulées de même format.

In [ ]:
#@title Chargement de kiloword (exécuter ; pas besoin de lire le code)
def charger_kiloword():
    try:
        chemin = mne.datasets.kiloword.data_path() / "kword_metadata-epo.fif"
        return mne.read_epochs(chemin, preload=True), "kiloword"
    except Exception as err:
        print("Téléchargement impossible, données simulées :", type(err).__name__)
        rng = np.random.default_rng(0)
        sfreq, times = 250.0, np.arange(-0.1, 0.924, 1 / 250)
        noms = ["Fz", "Cz", "Pz", "C3", "C4", "P3", "P4", "O1", "O2", "F3", "F4"]
        n = 960
        meta = pd.DataFrame({
            "WORD": [f"mot{i}" for i in range(n)],
            "Concreteness": rng.uniform(1, 7, n),
            "WordFrequency": rng.uniform(0, 4, n),
            "NumberOfLetters": rng.integers(3, 10, n).astype(float),
        })
        n400 = np.exp(-((times - 0.4) ** 2) / (2 * 0.07 ** 2))
        ampl = (-2 + 0.8 * meta["WordFrequency"].to_numpy())[:, None] * 1e-6
        data = ampl[:, None, :] * n400 + 1.5e-6 * rng.standard_normal((n, len(noms), len(times)))
        info = mne.create_info(noms, sfreq, "eeg")
        ep = mne.EpochsArray(data, info, tmin=times[0], metadata=meta)
        ep.set_montage("standard_1020")
        return ep, "simulé"

epochs_kw, source = charger_kiloword()
print("Source :", source)
print(epochs_kw)

### Bloc Type 1 — Recoller les essais et ajouter du bruit

La cellule suivante :

1. place les 960 essais bout à bout ;
2. ajoute une **dérive lente** (0,03 Hz, comme la transpiration) et le **bruit du secteur** à 60 Hz ;
3. marque le début de chaque mot par une **annotation**, `mot/freq_faible` ou `mot/freq_elevee` selon la fréquence lexicale.

Le résultat est un objet `Raw`, comme celui qu'on obtient en ouvrant un fichier EEG.

In [ ]:
#@title Construction du signal continu (exécuter ; pas besoin de lire le code)
donnees = epochs_kw.get_data()                         # (essais, canaux, temps), en volts
n_essais, n_canaux, n_temps = donnees.shape
sfreq = epochs_kw.info["sfreq"]

continu = np.concatenate(list(donnees), axis=1)        # (canaux, essais x temps)
t = np.arange(continu.shape[1]) / sfreq

rng = np.random.default_rng(2)
derive = 8e-6 * np.sin(2 * np.pi * 0.03 * t + rng.uniform(0, 2 * np.pi, (n_canaux, 1)))
secteur = 3e-6 * np.sin(2 * np.pi * 60 * t)
continu = continu + derive + secteur

raw = mne.io.RawArray(continu, epochs_kw.info.copy())

meta = epochs_kw.metadata.reset_index(drop=True)
mediane = meta["WordFrequency"].median()
description = np.where(meta["WordFrequency"] < mediane, "mot/freq_faible", "mot/freq_elevee")
idx0 = np.argmin(np.abs(epochs_kw.times))              # échantillon du temps 0 dans un essai
onsets = (np.arange(n_essais) * n_temps + idx0) / sfreq
raw.set_annotations(mne.Annotations(onset=onsets, duration=0, description=description))
print(raw)

## 2. `Raw` et `info`

### Bloc Type 1 — Ce qu'il faut vérifier en ouvrant un fichier

In [ ]:
# ---- Informations principales
print("Fréquence d'échantillonnage :", raw.info["sfreq"], "Hz")
print("Nombre de canaux            :", len(raw.ch_names))
print("Premiers canaux             :", raw.ch_names[:6])
print("Durée                       : %.1f s" % raw.times[-1])
print("Filtres déjà appliqués      :", raw.info["highpass"], "à", raw.info["lowpass"], "Hz")

<details><summary>Tout le contenu de <code>info</code></summary>

`print(raw.info)` affiche l'ensemble : canaux, types, positions, filtres, date d'enregistrement, ...
</details>

In [ ]:
# ---- Position des électrodes
raw.plot_sensors(show_names=True);

### Bloc Type 1 — Voir le signal

Les traits verticaux sont les annotations. `scalings` fixe l'échelle (20 µV), `start` le début en secondes.

In [ ]:
# ---- 10 secondes de signal
raw.plot(start=100, duration=10, n_channels=15, scalings=dict(eeg=20e-6));

<details><summary>Figure fixe ou interactive ?</summary>

Dans un notebook, la figure est fixe. Dans un script lancé depuis le terminal, elle est interactive : défilement, zoom, marquage des mauvais canaux.
</details>

### Bloc Type 2 — À vous de compléter

1. Combien d'échantillons contient le signal ? (`raw.n_times`)
2. Créez `raw_centre` avec seulement `Fz`, `Cz`, `Pz`, sans modifier `raw` : `raw.copy().pick([...])`.
3. Affichez 5 s de `raw_centre` à partir de 300 s.

In [ ]:
# ---- Exercice
# print(...)                                     # <--- à compléter
# raw_centre = ...                               # <--- à compléter
# raw_centre.plot(...)                           # <--- à compléter

In [ ]:
#@title Solution (à consulter après avoir essayé)
print("Échantillons :", raw.n_times)
raw_centre = raw.copy().pick(["Fz", "Cz", "Pz"])
raw_centre.plot(start=300, duration=5, scalings=dict(eeg=20e-6));

## 3. Spectre de puissance et filtrage

### Bloc Type 1 — Spectre de puissance (PSD)

Puissance du signal à chaque fréquence (axe logarithmique). Repérez les deux bruits ajoutés : le **pic vers 0,03 Hz** (dérive) et le **pic à 60 Hz** (secteur).

`n_fft` fixe la fenêtre d'analyse : 100 s, pour distinguer des fréquences aussi basses. Les pics fins vers 1, 2, 3 Hz, ... viennent de la construction du signal (un mot toutes les 1,024 s).

In [ ]:
# ---- PSD du signal non filtré
spectre = raw.compute_psd(fmin=0.01, fmax=sfreq / 2, n_fft=int(100 * sfreq))
spectre.plot(average=True, amplitude=False, xscale="log");

### Bloc Type 1 — Filtrer

| Fonction | Effet |
|---|---|
| `raw.filter(l_freq=0.5, h_freq=30)` | passe-bande : retire la dérive (sous 0,5 Hz) et les hautes fréquences (au-dessus de 30 Hz) |
| `raw.notch_filter(60)` | coupe-bande : retire seulement une bande étroite autour de 60 Hz |

Les filtres modifient les données : on filtre une **copie** (`raw.copy()`) pour garder `raw` intact.

In [ ]:
# ---- Deux versions filtrées
raw_notch = raw.copy().notch_filter(freqs=60)
raw_filtre = raw.copy().filter(l_freq=0.5, h_freq=30)

In [ ]:
#@title Figure : spectres avant et après filtrage (exécuter ; pas besoin de lire le code)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for ax, (titre, r) in zip(axes, [("Non filtré", raw), ("Coupe-bande 60 Hz", raw_notch),
                                 ("Passe-bande 0,5-30 Hz", raw_filtre)]):
    r.compute_psd(fmin=0.01, fmax=sfreq / 2, n_fft=int(100 * sfreq)).plot(average=True, amplitude=False, xscale="log", axes=ax, show=False)
    ax.set_title(titre)
plt.tight_layout()
plt.show()

In [ ]:
#@title Figure : canal Pz avant et après filtrage, 10 s (exécuter ; pas besoin de lire le code)
debut, fin = raw.time_as_index([100, 110])
x = raw.times[debut:fin]
plt.figure(figsize=(10, 3))
plt.plot(x, raw.get_data(picks="Pz")[0, debut:fin] * 1e6, label="non filtré", lw=0.8)
plt.plot(x, raw_filtre.get_data(picks="Pz")[0, debut:fin] * 1e6, label="0,5-30 Hz", lw=1.2)
plt.xlabel("Temps (s)"); plt.ylabel("µV"); plt.legend()
plt.show()

### Bloc Type 2 — À vous de compléter

1. Passe-haut à **0,1 Hz** au lieu de 0,5 Hz : la dérive est-elle toujours retirée ?
2. Passe-bas à **40 Hz** seulement (`l_freq=None`) : le pic à 60 Hz disparaît-il ?

In [ ]:
# ---- Exercice
l_freq_test = 0.5                                # <--- essayez 0.1
h_freq_test = 30                                 # <--- essayez 40, avec l_freq_test = None

raw_test = raw.copy().filter(l_freq=l_freq_test, h_freq=h_freq_test)
raw_test.compute_psd(fmin=0.01, fmax=sfreq / 2, n_fft=int(100 * sfreq)).plot(average=True, amplitude=False, xscale="log");

In [ ]:
#@title Solution (à consulter après avoir essayé)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
raw.copy().filter(0.1, 30).compute_psd(fmin=0.01, fmax=sfreq / 2, n_fft=int(100 * sfreq)).plot(average=True, amplitude=False, xscale="log", axes=axes[0], show=False)
raw.copy().filter(None, 40).compute_psd(fmin=0.01, fmax=sfreq / 2, n_fft=int(100 * sfreq)).plot(average=True, amplitude=False, xscale="log", axes=axes[1], show=False)
axes[0].set_title("0,1-30 Hz"); axes[1].set_title("passe-bas 40 Hz")
plt.tight_layout(); plt.show()

### Bloc Type 3 — Pour aller plus loin

Un passe-haut trop élevé (au-dessus de 1 Hz environ) peut déformer les composantes lentes comme la N400. Voir Tanner, Morgan-Short et Luck (2015), *Psychophysiology*, 52(8), 997-1009.

## 4. Événements et `Epochs`

### Bloc Type 1 — Des annotations aux événements

MNE découpe les données à partir d'un tableau d'**événements** : une ligne par stimulus, `[échantillon, 0, code]`.

In [ ]:
# ---- Tableau d'événements
events, event_id = mne.events_from_annotations(raw_filtre)
print(event_id)
print(events[:5])

In [ ]:
# ---- Les événements dans le temps
mne.viz.plot_events(events, sfreq=sfreq, event_id=event_id);

### Bloc Type 1 — Découper en epochs

- `tmin`, `tmax` : fenêtre autour de chaque stimulus (en secondes).
- `baseline=(None, 0)` : soustrait la moyenne avant le stimulus.
- `metadata` : une ligne par mot, pour sélectionner les essais.

In [ ]:
# ---- Raw filtré -> Epochs
epochs = mne.Epochs(raw_filtre, events, event_id,
                    tmin=-0.1, tmax=0.9, baseline=(None, 0),
                    metadata=meta, preload=True)
print(epochs)

In [ ]:
# ---- Sélectionner une condition par une partie de son nom
print(epochs["freq_faible"])

### Bloc Type 1 — Tous les essais d'un coup d'œil

Une ligne par essai (couleur = amplitude), la moyenne en dessous. Utile pour repérer des essais anormaux.

In [ ]:
# ---- Canal Pz
epochs.plot_image(picks="Pz", vmin=-10, vmax=10);

### Bloc Type 2 — À vous de compléter

1. Epochs de -0,2 à 0,8 s **sans** ligne de base (`baseline=None`).
2. Nombre d'essais de chaque condition (`len(...)`).
3. Mots de plus de 7 lettres : `epochs_test["NumberOfLetters > 7"]`.

In [ ]:
# ---- Exercice
# epochs_test = mne.Epochs(...)                  # <--- à compléter
# print(len(...), len(...))                      # <--- à compléter
# longs = ...                                    # <--- à compléter

In [ ]:
#@title Solution (à consulter après avoir essayé)
epochs_test = mne.Epochs(raw_filtre, events, event_id, tmin=-0.2, tmax=0.8,
                         baseline=None, metadata=meta, preload=True)
print("freq_faible :", len(epochs_test["freq_faible"]), "| freq_elevee :", len(epochs_test["freq_elevee"]))
longs = epochs_test["NumberOfLetters > 7"]
print("Mots de plus de 7 lettres :", len(longs))

## 5. `Evoked` et cartes topographiques

### Bloc Type 1 — Moyenner les essais

`epochs.average()` renvoie un `Evoked` : l'ERP. La couleur de chaque courbe indique la position de l'électrode.

In [ ]:
# ---- ERP de tous les mots
evoked = epochs.average()
evoked.plot(spatial_colors=True, gfp=True);

### Bloc Type 1 — Cartes topographiques

L'amplitude sur le scalp à un instant donné. `average=0.05` : moyenne sur 50 ms autour de chaque instant.

In [ ]:
# ---- Cartes de 100 à 500 ms
evoked.plot_topomap(times=[0.1, 0.2, 0.3, 0.4, 0.5], average=0.05, size=1.5);

In [ ]:
# ---- Courbes et cartes ensemble
evoked.plot_joint(times=[0.2, 0.4], title="Tous les mots");

### Bloc Type 1 — Comparer deux conditions

Mots de fréquence faible vs élevée (même comparaison que dans `seance4_outils_python.ipynb`, cette fois avec MNE). La **différence** isole l'effet de la fréquence.

Dans ces données, les deux courbes restent proches entre 300 et 500 ms : l'effet de la fréquence sur la N400 est faible à Pz. La concrétude (exercice ci-dessous) donne un effet plus net.

In [ ]:
# ---- ERP par condition à Pz
evokeds = {
    "fréquence faible": epochs["freq_faible"].average(),
    "fréquence élevée": epochs["freq_elevee"].average(),
}
mne.viz.plot_compare_evokeds(evokeds, picks="Pz", invert_y=True, show_sensors=False);

In [ ]:
# ---- Onde de différence (faible - élevée)
difference = mne.combine_evoked([evokeds["fréquence faible"], evokeds["fréquence élevée"]], weights=[1, -1])
difference.plot_topomap(times=[0.3, 0.4, 0.5], average=0.1, size=2);

### Bloc Type 2 — À vous de compléter

1. Carte de `evoked` **moyennée sur la fenêtre P200** (0,15 à 0,25 s). Indice : un instant au centre, `average` = largeur de la fenêtre.
2. Même comparaison avec la **concrétude** (`Concreteness`, coupe à la médiane).

In [ ]:
# ---- Exercice
# evoked.plot_topomap(times=..., average=...)       # <--- à compléter

# med = ...                                         # <--- à compléter
# evokeds_conc = {"abstrait": ..., "concret": ...}  # <--- à compléter
# mne.viz.plot_compare_evokeds(evokeds_conc, picks="Pz", invert_y=True, show_sensors=False);

In [ ]:
#@title Solution (à consulter après avoir essayé)
evoked.plot_topomap(times=0.2, average=0.1, size=2);

med = epochs.metadata["Concreteness"].median()
evokeds_conc = {
    "abstrait": epochs[f"Concreteness < {med}"].average(),
    "concret": epochs[f"Concreteness >= {med}"].average(),
}
mne.viz.plot_compare_evokeds(evokeds_conc, picks="Pz", invert_y=True, show_sensors=False);
mne.combine_evoked([evokeds_conc["concret"], evokeds_conc["abstrait"]], weights=[1, -1]).plot_topomap(
    times=[0.3, 0.4, 0.5], average=0.1, size=2);

### Bloc Type 3 — Pour aller plus loin

1. Enregistrez et relisez : `epochs.save("kiloword-epo.fif")`, `evoked.save("kiloword-ave.fif")`, puis `mne.read_epochs(...)`, `mne.read_evokeds(...)`.
2. Comparez avec l'ERP d'origine (`epochs_kw.average()`) : le filtrage a-t-il modifié la N400 ?

## 6. Récapitulatif

| MNE | Rôle |
|---|---|
| `Raw`, `raw.info` | signal continu et sa description |
| `raw.plot`, `raw.plot_sensors` | inspection du signal et des électrodes |
| `raw.compute_psd` | repérer le bruit, choisir les filtres |
| `raw.filter`, `raw.notch_filter` | filtrage (sur une copie) |
| `events_from_annotations`, `mne.Epochs` | découpage autour des stimuli |
| `epochs.average()` → `Evoked` | ERP ; `combine_evoked` pour les différences |
| `plot_topomap`, `plot_joint`, `plot_compare_evokeds` | cartes et comparaisons |